# Attack on MEDS GGM path building using CW

# Clock glitching

## Setup

In [69]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEARM'
SS_VER = 'SS_VER_2_1'

In [70]:
%run "Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍
scope.clock.adc_freq                     changed from 30128391                  to 29538459                 
scope.clock.adc_rate                     changed from 30128391.0                to 29538459.0               
scope.io.hs2                             changed from glitch                    to clkgen                   


In [146]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../firmware/mcu/meds-glitch
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
arm-none-eabi-gcc (15:13.2.rel1-2) 13.2.1 20231009
Copyright (C) 2023 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWLITEARM 
.
Welcome to another exciting ChipWhisperer target build!!
.
.
.
Compiling:
Compiling:
Compiling:
-en     meds-glitch.c ...
.
-en     seed.c ...
.
-en     fips202.c ...
Compiling:
-en     .././simpleserial/simpleserial.c ...
Compiling:
-en     .././hal/hal.c ...
.
.
Compiling:
Compiling:
.
-en     .././hal//stm32f3/stm32f3_hal_lowlevel.c ...
-en     .././hal//stm32f3/stm32f3_hal.c ...
Compiling:
.
Assembling: .././hal//stm32f3/stm32f3_startup.S
arm-none-eabi-gcc -c -mcpu=cortex-m4 -I. -x assembler-with-cpp -mthumb -mfloat-abi=soft -fmessage-length=0 -ffunction-sections -DF_CPU=7372800 -Wa,-gstabs,-adhlns=objdir-CWLITEARM/stm32f3_startup.lst -I.././simpleserial/ 

seed.c: In function 'stree_to_path_to_stree_glitch':
seed.c:143:3: warning: implicit declaration of function 'trigger_high' [-Wimplicit-function-declaration]
  143 |   trigger_high();
      |   ^~~~~~~~~~~~
seed.c:180:5: warning: implicit declaration of function 'trigger_low' [-Wimplicit-function-declaration]
  180 |     trigger_low();
      |     ^~~~~~~~~~~


-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
.
LINKING:
-en     meds-glitch-CWLITEARM.elf ...
Memory region         Used Size  Region Size  %age Used
             RAM:        2480 B        40 KB      6.05%
             ROM:       10952 B       256 KB      4.18%
-e Done!
.
.
.
Creating load file for Flash: meds-glitch-CWLITEARM.hex
arm-none-eabi-objcopy -O ihex -R .eeprom -R .fuse -R .lock -R .signature meds-glitch-CWLITEARM.elf meds-glitch-CWLITEARM.hex
.
Creating load file for Flash: meds-glitch-CWLITEARM.bin
arm-none-eabi-objcopy -O binary -R .eeprom -R .fuse -R .lock -R .signature meds-glitch-CWLITEARM.elf meds-glitch-CWLITEARM.bin
Creating load file for EEPROM: meds-glitch-CWLITEARM.eep
arm-none-eabi-objcopy -j .eeprom --set-section-flags=.eeprom="alloc,load" \
--change-section-lma .eeprom=0 --no-change-warnings -O ihex meds-glitch-CWLITEARM.elf meds-glitch-CWLITEARM.eep || exit 0
Creating Extended Listing: meds-glitch-CWLITEARM.lss
arm-none-eabi-objdump 

In [147]:
fw_path = "../firmware/mcu/meds-glitch/meds-glitch-{}.hex".format(PLATFORM)
#prog = cw.programmers.STM32FProgrammer
cw.program_target(scope, prog, fw_path)
if SS_VER=='SS_VER_2_1':
    target.reset_comms()

Detected known STMF32: STM32F302xB(C)/303xB(C)
Extended erase (0x44), this can take ten seconds or more
Attempting to program 10951 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 10951 bytes


In [148]:
if PLATFORM == "CWLITEXMEGA":
    def reboot_flush():            
        scope.io.pdic = False
        time.sleep(0.1)
        scope.io.pdic = "high_z"
        time.sleep(0.1)
        #Flush garbage too
        target.flush()
else:
    def reboot_flush():            
        scope.io.nrst = False
        time.sleep(0.05)
        scope.io.nrst = "high_z"
        time.sleep(0.05)
        #Flush garbage too
        target.flush()

## Communication tests

In [149]:
msg = bytearray([0]) 
target.simpleserial_write('f', msg)
print(target.simpleserial_read('r', 1))

CWbytearray(b'42')


In [150]:
msg = bytearray([0]*16) 
golden_path = []
for i in range(7) :
    target.simpleserial_write('b', msg)
    golden_path.append(target.simpleserial_read('r', 16))
golden_path

[CWbytearray(b'9d 28 d3 c7 12 db b7 02 4d d7 7b 9a af 9e cf a2'),
 CWbytearray(b'c7 dc 3f ce 80 e7 47 b3 f5 62 65 1d 65 fc a7 6c'),
 CWbytearray(b'bc 12 fa 33 63 e0 f5 30 cf bd 03 50 99 2e 08 59'),
 CWbytearray(b'49 65 9a 22 55 74 29 3a 4f ef 3b 01 c5 a3 c9 a8'),
 CWbytearray(b'86 3c b7 48 45 f6 3c 35 60 1e d0 b2 68 99 05 29'),
 CWbytearray(b'7c aa 64 8f e4 91 72 99 f2 0b 1b e2 e9 71 2d 62'),
 CWbytearray(b'2c e6 70 86 17 13 a4 4a df 2e 00 03 9e c6 7d 92')]

In [151]:
def get_path() :
    msg = bytearray([0]*16) 
    faulted_path = [];
    for i in range(16) :
        target.simpleserial_write('g', msg)
        faulted_path.append(target.simpleserial_read('r', 16))
    return faulted_path
    
def full_path(faulted_path, reference_list):
    """
    Returns true if all the elements of the golden path are in the faulted one
    """
    for item in reference_list:
        if item not in faulted_path:
            print("{:02x} ".format(item[0]))
            return False
    return True

def get_diff(faulted_path, reference_list):
    """
    Counts CWByteArrays in faulted_path that are:
    1. Not a 16-byte null array (all zeros)
    2. Not present in the reference_list
    """
    NULL_SEED = bytearray([0]*16)
    
    diff_count = 0
    for item in faulted_path:
        if item == NULL_SEED:
            continue
        if item not in reference_list:
            diff_count += 1
            
    return diff_count

def pp_path(path) : 
    NULL_SEED = bytearray([0]*16)
    s = "[ "
    for item in path:
        if item == NULL_SEED:
            continue
        s += "{:02x} ".format(item[0])
    s += "]"
    return s

def pp_path_list(l) :
    print("[")
    for path in l :
        print(pp_path(path))
    print("]")

In [152]:
msg = bytearray([0])
target.simpleserial_write('f', msg)
val = target.simpleserial_read_witherrors('r', 1)
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']

print(val['full_response'])
print(val)

fp = get_path()
get_diff(fp, golden_path)


CWbytearray(b'00 72 01 42 3d 00')
{'valid': True, 'payload': CWbytearray(b'42'), 'full_response': CWbytearray(b'00 72 01 42 3d 00'), 'rv': bytearray(b'\x00')}


0

## Matrix export

## Glitch configuration

In [161]:
gc = cw.GlitchController(groups=["success", "reset", "normal", "non_unique"], parameters=["width", "offset", "tries"])

In [162]:
gc.display_stats()

IntText(value=0, description='success count:', disabled=True)

IntText(value=0, description='reset count:', disabled=True)

IntText(value=0, description='normal count:', disabled=True)

IntText(value=0, description='non_unique count:', disabled=True)

FloatSlider(value=0.0, continuous_update=False, description='width setting:', disabled=True, max=10.0, readout…

FloatSlider(value=0.0, continuous_update=False, description='offset setting:', disabled=True, max=10.0, readou…

FloatSlider(value=0.0, continuous_update=False, description='tries setting:', disabled=True, max=10.0, readout…

In [163]:
gc.glitch_plot(plotdots={ "reset":"xr", "non_unique":"*b", "success":"+g", "normal":None})

:DynamicMap   []
   :Overlay
      .Points.I   :Points   [width,offset]
      .Points.II  :Points   [width,offset]
      .Points.III :Points   [width,offset]

In [164]:
#Basic setup
# set glitch clock
scope.glitch.clk_src = "clkgen" 
scope.glitch.output = "clock_xor" # glitch_out = clk ^ glitch
scope.glitch.trigger_src = "ext_single" # glitch only after scope.arm() called
scope.io.hs2 = "glitch"  # output glitch_out on the clock line
print(scope.glitch)

clk_src     = clkgen
mmcm_locked = True
width       = 10.15625
width_fine  = 0
offset      = 1.171875
offset_fine = 0
trigger_src = ext_single
arm_timing  = after_scope
ext_offset  = 4
repeat      = 2
output      = clock_xor



### Settings

In [165]:
import chipwhisperer.common.results.glitch as glitch
from tqdm.notebook import trange
import struct

scope.glitch.ext_offset = 4

gc.set_range("width", 4, 10)
gc.set_range("offset", -1, 1)
gc.set_global_step([1])

scope.glitch.repeat = 10
gc.set_step("tries", 1)
scope.adc.timeout = 0.1

for glitch_setting in gc.glitch_values():
    print("offset: {:4.1f}; width: {:4.1f}".format(glitch_setting[1], glitch_setting[0]))
reboot_flush()

offset: -1.0; width:  4.0
offset: -1.0; width:  4.0
offset: -1.0; width:  4.0
offset: -1.0; width:  4.0
offset: -1.0; width:  4.0
offset: -1.0; width:  4.0
offset: -1.0; width:  4.0
offset: -1.0; width:  4.0
offset: -1.0; width:  4.0
offset: -1.0; width:  4.0
offset: -1.0; width:  4.0
offset:  0.0; width:  4.0
offset:  0.0; width:  4.0
offset:  0.0; width:  4.0
offset:  0.0; width:  4.0
offset:  0.0; width:  4.0
offset:  0.0; width:  4.0
offset:  0.0; width:  4.0
offset:  0.0; width:  4.0
offset:  0.0; width:  4.0
offset:  0.0; width:  4.0
offset:  0.0; width:  4.0
offset:  1.0; width:  4.0
offset:  1.0; width:  4.0
offset:  1.0; width:  4.0
offset:  1.0; width:  4.0
offset:  1.0; width:  4.0
offset:  1.0; width:  4.0
offset:  1.0; width:  4.0
offset:  1.0; width:  4.0
offset:  1.0; width:  4.0
offset:  1.0; width:  4.0
offset:  1.0; width:  4.0
offset: -1.0; width:  5.0
offset: -1.0; width:  5.0
offset: -1.0; width:  5.0
offset: -1.0; width:  5.0
offset: -1.0; width:  5.0
offset: -1.0

In [166]:
faulted_trees = []
other_faults = []

## Glitch loop

In [167]:
found = 0
def glitch_loop() :
    for glitch_setting in gc.glitch_values():
        
        # optional: you can speed up the loop by checking if the trigger never went low
        #           (the target never called trigger_low();) via scope.adc.state
        scope.glitch.offset = glitch_setting[1]
        scope.glitch.width = glitch_setting[0]
    
        scope.arm()
        
        target.simpleserial_write('f', bytearray([0]))
        
        ret = scope.capture()
        
        val = target.simpleserial_read_witherrors('r', 1)#For loop check
        
        # ###################
        # Add your code here
        # ###################
        
        if ret: #here the trigger never went high - sometimes the target is still crashed from a previous glitch
            print('Timeout - no trigger')
            gc.add("reset")
    
            #Device is slow to boot?
            reboot_flush()
        else:
            # diff_num is the number of changed matrix values
            
            if val['valid'] == False :
                gc.add('reset')
                print("invalid response")
            else :
                faulted_path = get_path()
                diff_num = get_diff(faulted_path, golden_path)
                full = full_path(faulted_path, golden_path)
                if diff_num == 0 and full: 
                    gc.add('normal')
                elif diff_num == 1 and full: 
                    gc.add('success')
                    faulted_trees.append(faulted_path)
                    print(pp_path(faulted_path))
                else :
                    print(diff_num)
                    gc.add('non_unique')
                    other_faults.append(faulted_path)
                    print(pp_path(faulted_path))
    return 0
                    

In [168]:
glitch_loop()
print(faulted_trees)
print(other_faults)

(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:731) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0f


invalid response


(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:731) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0e


Timeout - no trigger


(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:731) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0f


...

Timeout - no trigger
invalid response


(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:731) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0f
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:731) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0e


Timeout - no trigger
invalid response


(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:731) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0f
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:731) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0c


Timeout - no trigger
[]
[]


In [145]:
pp_path_list(faulted_trees)
pp_path_list(other_faults)

[
]
[
[ ab ]
[ ab ]
[ ab ]
[ ab ]
[ ]
[ ab ]
[ ab ]
[ ab ]
[ ab ]
[ ab ]
[ ab ]
[ ab ]
[ ab ]
[ ab ]
[ ab ]
[ ab ]
[ ab ]
[ ]
[ ]
[ ]
[ ab ]
[ ab ]
[ ab ]
]


In [35]:
print(pp_path(golden_path))

[ 9d c7 bc 49 86 7c 2c ]


## Results

In [ ]:
results = gc.calc(ignore_params="tries", sort="success_rate")
results

## Disconnect

In [ ]:
scope.dis()
target.dis()